# Shareholder Letters 2000–2012 — Q&A Dataset Generation

**Source:** `Warren-Buffett-Berkshire-Letters-1957-2012.pdf` (pages 811–1094 only)  
**Content:** 13 annual shareholder letters covering the dot-com aftermath, financial crisis, and recovery era. Each letter is cut at Buffett's sign-off (`Warren E. Buffett / Chairman of the Board`) to exclude the financial appendix that follows.

**Why these years matter:** The 2000–2012 period contains Buffett's most explicit writing on Adaptability (defending his approach during dot-com mania, deploying capital during the 2008 crisis, evolving from domestic to international investments) and Personal Life (increasingly reflective tone as he ages, succession planning, philanthropic decisions). These are the two labels with the largest gaps in the current dataset.

**Pipeline:** Three-prompt generation (reference, conceptual, analytical) produces varied answer lengths from each chunk. 266 chunks × 3 passes = ~798 raw candidates before quality filtering.

In [1]:
import sys, re, statistics, importlib
from pathlib import Path
from collections import Counter

sys.path.insert(0, "..")
import pipeline.core
importlib.reload(pipeline.core)
from pipeline.core import (
    Chunk, QAPair, LABELS, extract_text,
    classify_chunks, generate_all, score_all,
    filter_by_quality, coverage_audit,
    export_csv, export_detailed,
    save_checkpoint, load_checkpoint,
)
import fitz

## Stage 1: Chunking

Letter boundaries detected via `BERKSHIRE HATHAWAY INC.` + `To the Shareholders` pattern at 13 known page positions. Each letter's text is truncated at Buffett's sign-off to exclude the financial appendix. Table paragraphs (>15% numeric content or dot-leader alignment) are filtered. Validated in `test_shareholder_chunking.py`.

In [2]:
BOUNDARY_PAGES = [811, 831, 849, 871, 893, 918, 940, 963, 984, 1006, 1025, 1052, 1073]


def extract_shareholder_letters(pdf_path):
    doc = fitz.open(pdf_path)
    letters = []
    for i, start_pg in enumerate(BOUNDARY_PAGES):
        end_pg = BOUNDARY_PAGES[i + 1] if i + 1 < len(BOUNDARY_PAGES) else len(doc)
        letter_text = ""
        for pg in range(start_pg, end_pg):
            letter_text += doc[pg].get_text()

        year_match = re.search(r'(?:during|in)\s+(\d{4})', letter_text[:500])
        if not year_match:
            year_match = re.search(r'\b(20[0-1]\d)\b', letter_text[:300])
        year = year_match.group(1) if year_match else "UNKNOWN"

        signoff = re.search(
            r'Warren E\. Buffett\s*\n\s*(?:February|March|April)\s+\d{1,2},?\s*\d{4}\s*\n\s*Chairman',
            letter_text
        )
        if not signoff:
            signoff = re.search(r'Warren E\. Buffett\s*\n.*?Chairman of the Board', letter_text, re.DOTALL)
        if not signoff:
            all_wb = list(re.finditer(r'Warren E\. Buffett', letter_text))
            if all_wb:
                signoff = all_wb[-1]

        buffett_text = letter_text[:signoff.start()].strip() if signoff else letter_text.strip()
        letters.append({"year": year, "text": buffett_text})
    doc.close()
    return letters


def is_financial_table(text, threshold=0.15):
    if len(text) < 50:
        return True
    noise = sum(1 for c in text if c.isdigit() or c in '$%.,') / len(text)
    dot_runs = len(re.findall(r'\.{3,}', text))
    return noise > threshold or dot_runs > 3


def _sub_chunk(text, source_file, section, max_chars=4000, overlap=500):
    if len(text) <= max_chars:
        return [Chunk(text=text, source_file=source_file,
                      source_section=section, chunk_strategy="shareholder_letter")]
    paragraphs = [p.strip() for p in text.split('\n') if len(p.strip()) > 30]
    sub_chunks, current, current_len = [], [], 0
    for para in paragraphs:
        if current_len + len(para) > max_chars and current:
            sub_chunks.append(Chunk(
                text='\n'.join(current), source_file=source_file,
                source_section=section, chunk_strategy="shareholder_subchunk"))
            overlap_paras, overlap_len = [], 0
            for p in reversed(current):
                if overlap_len + len(p) > overlap: break
                overlap_paras.insert(0, p)
                overlap_len += len(p)
            current = overlap_paras + [para]
            current_len = overlap_len + len(para)
        else:
            current.append(para)
            current_len += len(para)
    if current:
        sub_chunks.append(Chunk(
            text='\n'.join(current), source_file=source_file,
            source_section=section,
            chunk_strategy="shareholder_subchunk" if sub_chunks else "shareholder_letter"))
    return sub_chunks


def chunk_shareholder_letters(pdf_path):
    letters = extract_shareholder_letters(pdf_path)
    all_chunks = []
    tables_filtered = 0

    for letter in letters:
        text = letter["text"]
        text = re.sub(r'^\d{1,4}\s*$', '', text, flags=re.MULTILINE)
        text = re.sub(r'^BERKSHIRE HATHAWAY INC\.\s*$', '', text, flags=re.MULTILINE)

        paragraphs = [p.strip() for p in text.split('\n') if len(p.strip()) > 30]
        clean = []
        for para in paragraphs:
            if is_financial_table(para):
                tables_filtered += 1
            else:
                clean.append(para)

        if not clean:
            continue

        clean_text = '\n'.join(clean)
        section_name = f"{letter['year']} Shareholder Letter"
        all_chunks.extend(_sub_chunk(clean_text, "shareholder_letters_2000_2012.pdf", section_name))

    merged = []
    for chunk in all_chunks:
        if len(chunk.text) < 1000 and merged:
            merged[-1].text += '\n' + chunk.text
        else:
            merged.append(chunk)

    print(f"Letters: {len(letters)} | Tables filtered: {tables_filtered} | Chunks: {len(merged)}")
    return merged

In [3]:
chunks = chunk_shareholder_letters(Path("../sources/Warren-Buffett-Berkshire-Letters-1957-2012.pdf"))

Letters: 13 | Tables filtered: 1521 | Chunks: 266


## Stage 2: Classification

266 chunks through DeepSeek. Crisis-year letters (2001, 2002, 2008, 2009) should produce significant Adaptability and Risk Management classifications. Later letters (2010–2012) contain increasing Personal Life content as Buffett reflects on succession and legacy.

In [4]:
classified = await classify_chunks(chunks)
save_checkpoint(classified, "shareholder_classified")

Classifying 266 chunks (skipping 0 pre-labeled)
  10/266
  20/266
  30/266
  40/266
  50/266
  60/266
  70/266
  80/266
  90/266
  100/266
  110/266
  120/266
  130/266
  140/266
  150/266
  160/266
  170/266
  180/266
  190/266
  200/266
  210/266
  220/266
  230/266
  240/266
  250/266
  260/266
  266/266

Label distribution:
  Strategy Development: 133
  Risk Management: 75
  Personal Life: 31
  Psychology: 15
  Adaptability: 9
  Timing: 3
Saved: C:\Users\Admin\OneDrive\Desktop\buffet-qa-pipeline\intermediate\shareholder_classified.pkl


In [5]:
dist = Counter(c.label for c in classified if c.label)
print("Label distribution:\n")
for label, count in dist.most_common():
    bar = "█" * count
    print(f"  {label:25s} {count:3d} {bar}")
failed = [c for c in classified if c.label is None]
if failed:
    print(f"\n!! {len(failed)} chunks unclassified")

Label distribution:

  Strategy Development      133 █████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  Risk Management            75 ███████████████████████████████████████████████████████████████████████████
  Personal Life              31 ███████████████████████████████
  Psychology                 15 ███████████████
  Adaptability                9 █████████
  Timing                      3 ███


## Stage 3: Q&A Generation

Three passes per chunk — reference (short factual), conceptual (reasoning), and analytical (deep multi-dimensional). Each pass uses a structurally different prompt so the LLM performs a fundamentally different task each time, guaranteeing answer length diversity. 266 chunks × 3 passes = 798 generation calls.

In [16]:
pairs = await generate_all(classified)
save_checkpoint(pairs, "shareholder_pairs_raw")
print(f"\nTotal raw pairs: {len(pairs)}")


  Pass: reference
    5/266 chunks | 5 total pairs
    10/266 chunks | 10 total pairs
    15/266 chunks | 15 total pairs
    20/266 chunks | 20 total pairs
    25/266 chunks | 25 total pairs
    30/266 chunks | 30 total pairs
    35/266 chunks | 35 total pairs
    40/266 chunks | 40 total pairs
    45/266 chunks | 45 total pairs
    50/266 chunks | 50 total pairs
    55/266 chunks | 55 total pairs
    60/266 chunks | 60 total pairs
    65/266 chunks | 65 total pairs
    70/266 chunks | 70 total pairs
    75/266 chunks | 75 total pairs
    80/266 chunks | 80 total pairs
    85/266 chunks | 85 total pairs
    90/266 chunks | 90 total pairs
    95/266 chunks | 95 total pairs
    100/266 chunks | 100 total pairs
    105/266 chunks | 105 total pairs
    110/266 chunks | 110 total pairs
    115/266 chunks | 115 total pairs
    120/266 chunks | 120 total pairs
    125/266 chunks | 125 total pairs
    130/266 chunks | 130 total pairs
    135/266 chunks | 135 total pairs
    140/266 chunks | 1

In [17]:
# Verify length diversity across prompt types
for ptype in ["reference", "conceptual", "analytical"]:
    type_pairs = [p for p in pairs if p.prompt_type == ptype]
    if type_pairs:
        lengths = [len(p.answer.split()) for p in type_pairs]
        print(f"  {ptype:12s}: {len(type_pairs):3d} pairs | avg {sum(lengths)//len(lengths):2d} words | range {min(lengths)}–{max(lengths)}")

  reference   : 266 pairs | avg 21 words | range 5–46
  conceptual  : 266 pairs | avg 114 words | range 66–194
  analytical  : 266 pairs | avg 172 words | range 116–228


In [18]:
seen_labels = set()
for p in pairs:
    if p.label in seen_labels: continue
    seen_labels.add(p.label)
    print(f"── {p.label} ──")
    print(f"  Q: {p.question}")
    print(f"  A: {p.answer[:250]}...")
    print()

── Strategy Development ──
  Q: What was the per-share book value gain for Berkshire Hathaway in 2000?
  A: The per-share book value gain for Berkshire Hathaway in 2000 was 6.5%....

── Risk Management ──
  Q: What is the cost of float that Berkshire Hathaway reported for its insurance operations in 2000?
  A: In 2000, Berkshire had an underwriting loss of $1.6 billion, which gave it a float cost of 6%....

── Adaptability ──
  Q: What did Warren Buffett say about GEICO's increased advertising spending in 2000?
  A: He said he was wrong, as the extra money did not produce a commensurate increase in inquiries and led to a sharp rise in per-policy acquisition cost....

── Psychology ──
  Q: What does Warren Buffett warn about CEOs who publicly predict high annual earnings growth rates like 15%?
  A: He warns they often become 'congenital optimists, or even charlatans,' and their predictions can lead to uneconomic maneuvers and accounting shenanigans to meet the targets....

── Personal L

## Stage 4: Quality Scoring & Filtering

Scoring evaluates richness relative to question complexity — a 2-sentence reference answer scores the same as a 7-sentence analytical answer if both are appropriately detailed for their question type. Composite threshold remains 0.7.

In [19]:
chunk_map = {c.chunk_id: c for c in classified}
scored = await score_all(pairs, chunk_map)
filtered = filter_by_quality(scored, threshold=0.7)
save_checkpoint(filtered, "shareholder_pairs_filtered")

  Scored 10/798
  Scored 20/798
  Scored 30/798
  Scored 40/798
  Scored 50/798
  Scored 60/798
  Scored 70/798
  Scored 80/798
  Scored 90/798
  Scored 100/798
  Scored 110/798
  Scored 120/798
  Scored 130/798
  Scored 140/798
  Scored 150/798
  Scored 160/798
  Scored 170/798
  Scored 180/798
  Scored 190/798
  Scored 200/798
  Scored 210/798
  Scored 220/798
  Scored 230/798
  Scored 240/798
  Scored 250/798
  Scored 260/798
  Scored 270/798
  Scored 280/798
  Scored 290/798
  Scored 300/798
  Scored 310/798
  Scored 320/798
  Scored 330/798
  Scored 340/798
  Scored 350/798
  Scored 360/798
  Scored 370/798
  Scored 380/798
  Scored 390/798
  Scored 400/798
  Scored 410/798
  Scored 420/798
  Scored 430/798
  Scored 440/798
  Scored 450/798
  Scored 460/798
  Scored 470/798
  Scored 480/798
  Scored 490/798
  Scored 500/798
  Scored 510/798
  Scored 520/798
  Scored 530/798
  Scored 540/798
  Scored 550/798
  Scored 560/798
  Scored 570/798
  Scored 580/798
  Scored 590/798
  Scor

In [20]:
scores = [p.composite_score for p in filtered if p.composite_score]
print(f"Pairs after filtering: {len(filtered)} / {len(pairs)} raw")
print(f"Drop rate: {(1 - len(filtered)/len(pairs))*100:.1f}%\n")
print(f"Score range: {min(scores):.2f} — {max(scores):.2f}")
print(f"Mean:   {statistics.mean(scores):.2f}")
print(f"Median: {statistics.median(scores):.2f}")

Pairs after filtering: 653 / 798 raw
Drop rate: 18.2%

Score range: 0.70 — 0.94
Mean:   0.84
Median: 0.85


## Stage 5: Coverage & Export

This document targets the two weakest labels (Adaptability and Personal Life) while contributing to all others. The three-prompt approach should also produce visibly different answer length distributions across prompt types.

In [21]:
print("Shareholder 2000-2012 contribution:\n")
report = coverage_audit(filtered)

Shareholder 2000-2012 contribution:

  Personal Life              69 pairs (10.6%)
    Sublabels hit: early_life, mentors, habits, personal_values, lifestyle
  Strategy Development      326 pairs (49.9%)
    Sublabels hit: value_investing_framework, margin_of_safety, competitive_moat, business_quality, circle_of_competence, capital_allocation, graham_influence
  Timing                      9 pairs ( 1.4%)
    Sublabels hit: entry_criteria, market_cycles
  Risk Management           186 pairs (28.5%)
    Sublabels hit: position_sizing, diversification, leverage_avoidance, permanent_loss, insurance_float, margin_of_safety_risk, concentration
  Adaptability               21 pairs ( 3.2%)
    Sublabels hit: crisis_response, strategy_evolution, mistake_correction, market_regime_shifts, new_opportunities
  Psychology                 42 pairs ( 6.4%)
    Sublabels hit: temperament, emotional_discipline, contrarian_thinking, fear_greed, independence, rationality

  Total: 653 pairs


In [22]:
export_csv(filtered, Path("../intermediate/shareholder_qa.csv"))
export_detailed(filtered, Path("../intermediate/shareholder_qa_detailed.csv"))

Exported 653 pairs to ..\intermediate\shareholder_qa.csv
  Strategy Development: 326
  Risk Management: 186
  Personal Life: 69
  Psychology: 42
  Adaptability: 21
  Timing: 9
Detailed export: ..\intermediate\shareholder_qa_detailed.csv


In [23]:
label_counts = Counter(p.label for p in filtered)
print(f"FINAL: {len(filtered)} pairs across {len(label_counts)} labels\n")
for label, count in label_counts.most_common():
    best = max((p for p in filtered if p.label == label), key=lambda p: p.composite_score)
    print(f"── {label} ({count} pairs, best: {best.composite_score:.2f}) ──")
    print(f"  Q: {best.question}")
    print(f"  A: {best.answer[:300]}")
    print()

FINAL: 653 pairs across 6 labels

── Strategy Development (326 pairs, best: 0.94) ──
  Q: How does Warren Buffett's analysis of the newspaper industry's historical 'economic heaven' and its subsequent decline illustrate the interconnected application of his core principles of competitive moat, business quality erosion, and the margin of safety, and what does this reveal about the dynamic nature of value investing?
  A: Buffett's analysis shows that the newspaper industry's historical strength was a perfect example of a competitive moat created by a 'circularity' where high circulation attracted advertisers, and more ads/news attracted readers, leading to a monopoly in one-paper cities. This moat, as he notes, allo

── Risk Management (186 pairs, best: 0.94) ──
  Q: In his analysis of municipal bond insurance, how does Warren Buffett connect the historical rarity of tax-exempt defaults to a future, unappreciated risk, and what specific behavioral shift does he predict that transforms th

In [14]:
# classified = load_checkpoint("shareholder_classified")
# pairs = load_checkpoint("shareholder_pairs_raw")
# filtered = load_checkpoint("shareholder_pairs_filtered")